# 🌸 Flower Classification — Transfer Learning (MobileNetV2)

Classifies 5 flower types: **rose, tulip, dandelion, sunflower, daisy**.

**Expected dataset layout** (matches your `archive` folder):
```
archive/
  train/
    rose/...
    tulip/...
    dandelion/...
    sunflower/...
    daisy/...
  test/
    image1.jpg, image2.jpg, ...   <- flat, no labels, no subfolders
```

Since `test/` has no labels, it can't be used to *measure* accuracy — instead the model is trained and
evaluated using a train/validation split carved out of `train/` only, and `test/` is used purely for
**inference** (predicting labels for unlabeled images) at the end of the notebook.

**How to use in Google Colab:**
1. Runtime → Change runtime type → GPU (T4 is fine).
2. Run cells top to bottom.
3. Cell 2 lets you either (a) upload `archive.zip` directly, or (b) load it from Google Drive. Pick whichever matches where your dataset is.
4. The notebook auto-detects both the `train` folder (containing the 5 class subfolders) and the `test` folder (flat images), so it doesn't matter if `archive` has extra nesting inside it (common with Kaggle downloads).


In [ ]:
# Cell 1 — Check GPU
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## Step 1 — Get the dataset onto the Colab VM

Use **ONE** of the two options below depending on where your `archive` dataset lives.


In [ ]:
# Cell 2a — OPTION A: Upload archive.zip directly from your computer
# Run this cell, click "Choose Files", and select archive.zip
# Skip this cell if you're using Option B (Google Drive) instead.

from google.colab import files
import zipfile, os

uploaded = files.upload()  # select archive.zip in the dialog

zip_name = list(uploaded.keys())[0]
extract_path = "/content/archive"
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)


In [ ]:
# Cell 2b — OPTION B: Load from Google Drive instead
# Skip this cell if you already used Option A above.
# Update DRIVE_ZIP_PATH to wherever archive.zip (or the archive folder) sits in your Drive.

from google.colab import drive
import zipfile, os

drive.mount('/content/drive')

DRIVE_ZIP_PATH = "/content/drive/MyDrive/archive.zip"  # <-- change this to your actual path
extract_path = "/content/archive"

if DRIVE_ZIP_PATH.endswith(".zip"):
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extracted to:", extract_path)
else:
    # If you already have an unzipped 'archive' folder in Drive, just point straight to it instead.
    extract_path = DRIVE_ZIP_PATH
    print("Using folder directly:", extract_path)


## Step 2 — Auto-detect the train folder (labeled) and test folder (unlabeled)

Kaggle zips often unpack into an extra nested folder (e.g. `archive/flowers/train/rose/...` instead of
`archive/train/rose/...`). This cell searches recursively for:
- **TRAIN_DIR** — the folder that directly contains the 5 class subfolders (`rose`, `tulip`, etc.)
- **TEST_DIR** — a folder, usually a sibling of `train`, that contains only flat image files (no class subfolders, no labels)


In [ ]:
# Cell 3 — Locate TRAIN_DIR (5 class subfolders) and TEST_DIR (flat unlabeled images)
import os, re

EXPECTED_CLASSES = {"rose", "tulip", "dandelion", "sunflower", "daisy"}
IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def normalize(s):
    return re.sub(r'[^a-z]', '', s.lower())

def find_train_dir_exact(root):
    """Find the folder whose immediate subfolders are exactly the 5 flower classes."""
    for dirpath, dirnames, filenames in os.walk(root):
        names = {d.lower() for d in dirnames}
        if EXPECTED_CLASSES.issubset(names):
            return dirpath
    return None

def find_train_dir_fuzzy(root, min_matches=4):
    """Fallback: match folder names loosely (handles plurals, capitalization, extra characters,
    e.g. 'Roses', 'SUN_FLOWER', 'Daisy '). Picks the directory with the most matching class folders."""
    best_dir, best_score = None, 0
    for dirpath, dirnames, filenames in os.walk(root):
        norm_dirnames = [normalize(d) for d in dirnames]
        score = 0
        for cls in EXPECTED_CLASSES:
            if any(cls in nd or nd in cls for nd in norm_dirnames if nd):
                score += 1
        if score > best_score:
            best_score, best_dir = score, dirpath
    if best_score >= min_matches:
        return best_dir
    return None

def find_test_dir(root, train_dir):
    """Find a folder that holds flat image files directly (no class subfolders) — typically named 'test'."""
    candidates = []
    for dirpath, dirnames, filenames in os.walk(root):
        if train_dir and (dirpath == train_dir or train_dir in dirpath):
            continue  # skip the train dir and anything inside it (e.g. its class subfolders)
        images_here = [f for f in filenames if f.lower().endswith(IMG_EXTENSIONS)]
        if images_here and not (EXPECTED_CLASSES & {d.lower() for d in dirnames}):
            candidates.append((dirpath, len(images_here)))
    if not candidates:
        return None
    named_test = [c for c in candidates if os.path.basename(c[0]).lower() == "test"]
    if named_test:
        return max(named_test, key=lambda c: c[1])[0]
    return max(candidates, key=lambda c: c[1])[0]

def print_tree(root, max_depth=4):
    root = root.rstrip(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth >= max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        label = os.path.basename(dirpath) or dirpath
        extra = f"  ({len(filenames)} files)" if filenames else ""
        print(f"{indent}{label}/{extra}")

TRAIN_DIR = find_train_dir_exact(extract_path)
if TRAIN_DIR is None:
    TRAIN_DIR = find_train_dir_fuzzy(extract_path)

if TRAIN_DIR is None:
    print(f"Could not auto-detect a folder containing the 5 flower classes under: {extract_path}")
    print("Here is the actual folder structure so you can see what's there:\n")
    print_tree(extract_path, max_depth=4)
    raise FileNotFoundError(
        "Auto-detection failed. Look at the tree printed above, then set TRAIN_DIR manually in a new "
        "cell, e.g.:  TRAIN_DIR = '/content/archive/train'   (it must be the folder that directly "
        "contains the 5 class subfolders), then re-run from Cell 4 onward."
    )

TEST_DIR = find_test_dir(extract_path, TRAIN_DIR)

print("TRAIN_DIR:", TRAIN_DIR)
print("Subfolders found:", sorted(os.listdir(TRAIN_DIR)))
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(cls_path):
        n = len([f for f in os.listdir(cls_path) if f.lower().endswith(IMG_EXTENSIONS)])
        print(f"  {cls}: {n} images")

print()
if TEST_DIR:
    n_test = len([f for f in os.listdir(TEST_DIR) if f.lower().endswith(IMG_EXTENSIONS)])
    print(f"TEST_DIR: {TEST_DIR}  ({n_test} unlabeled images)")
else:
    print("No flat unlabeled test folder found — that's fine, the unlabeled-inference cell at the end will just be skipped.")


## Step 3 — Imports and config

In [ ]:
# Cell 4 — Imports
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 12        # training with the base model frozen
EPOCHS_FINE_TUNE = 10   # training with top layers of the base model unfrozen
VALIDATION_SPLIT = 0.2


## Step 4 — Load the dataset (train/validation split)

In [ ]:
# Cell 5 — Build tf.data datasets directly from the folder structure
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

class_names = train_ds.class_names
print("Class names (index order used by the model):", class_names)
num_classes = len(class_names)


In [ ]:
# Cell 6 — Performance: cache + prefetch
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Val batches:", tf.data.experimental.cardinality(val_ds).numpy())


In [ ]:
# Cell 7 — Visualize a few sample images to confirm labels look correct
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_idx = tf.argmax(labels[i]).numpy()
        plt.title(class_names[label_idx])
        plt.axis("off")
plt.tight_layout()
plt.show()


## Step 5 — Build the model (transfer learning with MobileNetV2)

MobileNetV2 pretrained on ImageNet is used as a frozen feature extractor first, then we fine-tune the top
layers for better accuracy. This generally gives much higher accuracy than training a CNN from scratch,
especially on a dataset this size.


In [ ]:
# Cell 8 — Data augmentation (applied only during training, inline in the model)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="data_augmentation")

# MobileNetV2 expects inputs preprocessed with its own preprocess_input (scales to [-1, 1])
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input


In [ ]:
# Cell 9 — Build the model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # freeze for the first training phase

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## Step 6 — Train (Phase 1: frozen base model)

In [ ]:
# Cell 10 — Callbacks + Phase 1 training
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint("best_model_phase1.keras", monitor="val_accuracy", save_best_only=True),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
)


## Step 7 — Fine-tune (Phase 2: unfreeze top layers of MobileNetV2)

Unfreezing the last ~30% of the base model's layers and training with a very low learning rate
typically gives a noticeable accuracy boost without overfitting too fast.


In [ ]:
# Cell 11 — Unfreeze top layers and continue training with a low LR
base_model.trainable = True

# Freeze everything except the last ~30% of layers
fine_tune_at = int(len(base_model.layers) * 0.7)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower LR for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

fine_tune_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint("best_model_finetuned.keras", monitor="val_accuracy", save_best_only=True),
]

total_epochs = EPOCHS_HEAD + EPOCHS_FINE_TUNE

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1,
    callbacks=fine_tune_callbacks,
)


## Step 8 — Plot training curves

In [ ]:
# Cell 12 — Combine and plot accuracy/loss across both training phases
acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.axvline(x=len(history.history['accuracy']) - 1, linestyle='--', color='gray', label='Fine-tune start')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.axvline(x=len(history.history['loss']) - 1, linestyle='--', color='gray', label='Fine-tune start')
plt.legend()
plt.title('Loss')

plt.tight_layout()
plt.show()


## Step 9 — Evaluate the model

Since `test/` has no labels, accuracy is measured on the held-out **validation split** of `train/`
(the 20% of labeled data the model never trained on). This is the honest accuracy number for this model.


In [ ]:
# Cell 13 — Evaluate: accuracy, confusion matrix, classification report (on the validation set)
val_loss, val_acc = model.evaluate(val_ds)
print(f"Validation accuracy: {val_acc:.4f}")

y_true = []
y_pred = []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(tf.argmax(labels, axis=1).numpy())
    y_pred.extend(tf.argmax(preds, axis=1).numpy())

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Validation Set)')
plt.show()


## Step 10 — Save the trained model

In [ ]:
# Cell 14 — Save the final model (and class name mapping) for later use
model.save("flower_classifier_final.keras")

import json as _json
with open("class_names.json", "w") as f:
    _json.dump(class_names, f)

print("Saved model to flower_classifier_final.keras")
print("Saved class names to class_names.json")

# Optional: download to your computer
# from google.colab import files
# files.download("flower_classifier_final.keras")
# files.download("class_names.json")


## Step 11 — Predict labels for the unlabeled `test/` folder

This is the real `test/` folder from your dataset (flat images, no ground-truth labels). The model predicts
a flower name for every image, shows a preview grid, and saves all predictions to a CSV file.


In [ ]:
# Cell 15 — Run inference on every image in TEST_DIR and save predictions to CSV
import csv
from tensorflow.keras.preprocessing import image as keras_image

if TEST_DIR is None:
    print("No TEST_DIR was detected — skipping this cell. See Cell 3 output.")
else:
    test_files = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(IMG_EXTENSIONS)])
    print(f"Found {len(test_files)} unlabeled images in {TEST_DIR}")

    results = []
    for fname in test_files:
        fpath = os.path.join(TEST_DIR, fname)
        img = keras_image.load_img(fpath, target_size=IMG_SIZE)
        img_array = keras_image.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_idx = int(np.argmax(preds))
        results.append({
            "filename": fname,
            "predicted_label": class_names[pred_idx],
            "confidence": float(preds[pred_idx]),
        })

    # Save all predictions to CSV
    with open("test_predictions.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["filename", "predicted_label", "confidence"])
        writer.writeheader()
        writer.writerows(results)
    print("Saved predictions for all", len(results), "images to test_predictions.csv")

    # Preview the first 9 predictions as a grid
    plt.figure(figsize=(10, 10))
    for i, r in enumerate(results[:9]):
        img = keras_image.load_img(os.path.join(TEST_DIR, r["filename"]), target_size=IMG_SIZE)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(img)
        plt.title(f"{r['predicted_label']} ({r['confidence']*100:.0f}%)")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

    # Optional: download the CSV to your computer
    # from google.colab import files
    # files.download("test_predictions.csv")


## Step 12 — Predict on any new photo you upload

Separate from the dataset's `test/` folder — upload any flower photo from your computer and the model
will show the image with its predicted label and confidence.


In [ ]:
# Cell 16 — Predict the flower type for a new uploaded image
from google.colab import files

def predict_flower(img_path, model, class_names):
    img = keras_image.load_img(img_path, target_size=IMG_SIZE)
    img_array = keras_image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)  # add batch dimension (no manual rescale — model does it internally)

    preds = model.predict(img_array, verbose=0)[0]
    pred_idx = np.argmax(preds)
    pred_label = class_names[pred_idx]
    confidence = preds[pred_idx] * 100

    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Predicted: {pred_label} ({confidence:.1f}% confidence)")
    plt.show()

    print("All class probabilities:")
    for name, prob in sorted(zip(class_names, preds), key=lambda x: -x[1]):
        print(f"  {name}: {prob*100:.2f}%")

    return pred_label, confidence

uploaded = files.upload()
for fname in uploaded.keys():
    predict_flower(fname, model, class_names)
